**Исследование данных и гипотезы о полезных преобразованиях**

Что вообще имеем

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic-Dataset.csv')

# Data audit
print("Размерность (строки, столбцы):", df.shape)

print("\n--- Первые 5 строк ---")
display(df.head())

print("\n--- Типы данных и заполненность ---")
df.info()

print("\n--- Статистика по числовым столбцам ---")
display(df.describe())

# Дубликаты: полностью одинаковых строк и уникальных PassengerId
print("\n--- Дубликаты ---")
print("Полных дубликатов строк:", df.duplicated().sum())
print("Уникальных PassengerId:", df["PassengerId"].nunique(), "из", len(df))

# Пропуски: сколько и в каких столбцах
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "dtype": df.dtypes
}).sort_values("missing_pct", ascending=False)

print("\n--- Пропуски по столбцам ---")
display(missing)

# Пропуски есть - это факт; надо что-то с ними сделать, а методов доступно несколько --> надо выбрать метод

Размерность (строки, столбцы): (891, 12)

--- Первые 5 строк ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



--- Типы данных и заполненность ---
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

--- Статистика по числовым столбцам ---


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200



--- Дубликаты ---
Полных дубликатов строк: 0
Уникальных PassengerId: 891 из 891

--- Пропуски по столбцам ---


,missing,missing_pct,dtype
Cabin,687,77.1,str
Age,177,19.9,float64
Embarked,2,0.2,str
PassengerId,0,0.0,int64
Name,0,0.0,str
Pclass,0,0.0,int64
Survived,0,0.0,int64
Sex,0,0.0,str
Parch,0,0.0,int64
SibSp,0,0.0,int64


Факты по пропускам:
- **Cabin** — много пропусков (~77%)
- **Age** — заметное количество (~20%)
- **Embarked** — единичные (2 строки)

Странность: сам факт пропуска в Cabin предсказывает выживание. Аномалия?

In [35]:
# Делим пассажиров на две группы по факту пропуска в Cabin и сравниваем
has_cabin = df["Cabin"].notna()

cabin_check = pd.DataFrame({
    "Человек": [has_cabin.sum(), (~has_cabin).sum()],
    "Доля выживших":    [df.loc[has_cabin, "Survived"].mean().round(2),       df.loc[~has_cabin, "Survived"].mean().round(2)],
    "Доля 1-го класса": [(df.loc[has_cabin, "Pclass"] == 1).mean().round(2), (df.loc[~has_cabin, "Pclass"] == 1).mean().round(2)],
    "Медианный Fare":    [df.loc[has_cabin, "Fare"].median().round(1),        df.loc[~has_cabin, "Fare"].median().round(1)],
}, index=["Каюта известна", "Каюта неизвестна"])
display(cabin_check)

print("Это MNAR (Missing Not At Random): вероятность пропуска зависит от самих данных")
print("сам факт «каюта известна» можно сохранить как признак (HasCabin).")

,Человек,Доля выживших,Доля 1-го класса,Медианный Fare
Каюта известна,204,0.67,0.86,55.2
Каюта неизвестна,687,0.30,0.06,10.5


Это MNAR (Missing Not At Random): вероятность пропуска зависит от самих данных
сам факт «каюта известна» можно сохранить как признак (HasCabin).


In [44]:
# Баланс классов целевой переменной
print("--- Survived: количество ---")
print(df["Survived"].value_counts())

print("\n--- Survived: доли ---")
print(df["Survived"].value_counts(normalize=True).round(2))

print("\nУмеренный дисбаланс классов: ~62% / 38%")

--- Survived: количество ---
Survived
0    549
1    342
Name: count, dtype: int64

--- Survived: доли ---
Survived
0    0.62
1    0.38
Name: proportion, dtype: float64

Умеренный дисбаланс классов: ~62% / 38%


In [37]:
# Кардинальность (cardinality) — число уникальных значений, от большего к меньшему
cardinality = df.nunique().sort_values(ascending=False)
for col, n in cardinality.items():
    print(col, "unique:", n)

print("\nНужно ли вообще считать Name, Ticket, Cabin как исходные категориальные признаки?")

PassengerId unique: 891
Name unique: 891
Ticket unique: 681
Fare unique: 248
Cabin unique: 147
Age unique: 88
SibSp unique: 7
Parch unique: 7
Embarked unique: 3
Pclass unique: 3
Survived unique: 2
Sex unique: 2

Нужно ли вообще считать Name, Ticket, Cabin как исходные категориальные признаки?


Числовые признаки и распределения

In [38]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

display(df.describe())

# Три графика в один ряд — удобнее смотреть на широком мониторе
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Возраст", "Цена билета", "box plot по цене билета"]
)
fig.add_trace(go.Histogram(x=df["Age"], marker_color="#4C78A8"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["Fare"], marker_color="#4C78A8"), row=1, col=2)
fig.add_trace(go.Box(y=df["Fare"], marker_color="#4C78A8", name="Fare"), row=1, col=3)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    height=400,
    title_text="Числовые признаки (numerical features)"
)
fig.show()

print("Выводы:")
print("- Fare (цена билета): сильный правый перекос — медиана 14, среднее 32, максимум 512;")
print("  box plot показывает много выбросов (outliers). Это надо чинить.")
print("- Age: форма распределения в целом нормальная, чинить её не надо,")
print("  ранее стало известно, что в Age 20% пропущено — это тотже надо чинить.")

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Выводы:
- Fare (цена билета): сильный правый перекос — медиана 14, среднее 32, максимум 512;
  box plot показывает много выбросов (outliers). Это надо чинить.
- Age: форма распределения в целом нормальная, чинить её не надо,
  ранее стало известно, что в Age 20% пропущено — это тотже надо чинить.


Изучаем категориальные признаки (categorical features)

In [45]:
# Три категориальных признака в один ряд
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Пол (Sex)", "Класс билета (Pclass)", "Порт посадки (Embarked)"])
fig.add_trace(go.Histogram(x=df["Sex"], marker_color="#4C78A8"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["Pclass"].astype(str), marker_color="#4C78A8"), row=1, col=2)
fig.add_trace(go.Histogram(x=df["Embarked"], marker_color="#4C78A8"), row=1, col=3)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    height=400,
    title_text="Категориальные признаки (categorical features)",
    title_x=0.5  # заголовок по центру
)
fig.update_yaxes(title_text="Количество человек", row=1, col=1)
fig.show()

print("Как распределён target внутри этих групп?")

# Доли выживших внутри каждого пола (normalize="index" — сумма по строке = 1)
ct = pd.crosstab(df["Sex"], df["Survived"], normalize="index").round(2)

# Значения male/female переводим на русский для подписей
sex_ru = df["Sex"].map({"male": "Мужчины", "female": "Женщины"})

# Таблица и гистограмма в один ряд (таблице отдаём 35% ширины, графику 65%)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "table"}, {"type": "xy"}]],
    column_widths=[0.35, 0.65],
    subplot_titles=["Доли внутри группы", "Количество человек"]
)
fig.add_trace(go.Table(
    header=dict(values=["Пол", "Погибли-0", "Выжили-1"], font=dict(size=16), height=32),
    cells=dict(values=[ct.index.map({"male": "Мужчины", "female": "Женщины"}), ct[0], ct[1]], font=dict(size=16), height=30)
), row=1, col=1)

# texttemplate="%{y}" — печатает абсолютное значение над каждым столбиком
fig.add_trace(go.Histogram(x=sex_ru[df["Survived"] == 0], name="Погибли", marker_color="#E45756", texttemplate="%{y}"), row=1, col=2)
fig.add_trace(go.Histogram(x=sex_ru[df["Survived"] == 1], name="Выжили", marker_color="#4C78A8", texttemplate="%{y}"), row=1, col=2)

fig.update_layout(
    template="plotly_white",
    barmode="group",
    height=400,
    title_text="Распределение выживших и погибших пассажиров по полу",
    title_x=0.5,  # заголовок по центру всей фигуры
    legend_title_text="Статус пассажира"
)
fig.update_xaxes(title_text="Пол", row=1, col=2)
fig.update_yaxes(title_text="Количество человек", row=1, col=2)
fig.show()

Как распределён target внутри этих групп?


Числовые признаки

In [46]:
# Survived в подписи и цвета
survived_ru = df["Survived"].map({0: "Погибли", 1: "Выжили"})
colors = {"Погибли": "#E45756", "Выжили": "#4C78A8"}

# Age по классам target: гистограмма + box plot сверху (marginal="box")
fig = px.histogram(
    df,
    x="Age",
    color=survived_ru,
    marginal="box",
    barmode="overlay",           # полупрозрачные гистограммы друг на друге — легко сравнивать форму
    color_discrete_map=colors,
    template="plotly_white",
    height=450,
    title="Распределение Age по классам target",
    labels={"color": "Статус пассажира", "Age": "Возраст"}
)
fig.update_layout(title_x=0.5, yaxis_title="Количество человек")
fig.show()

# Box plots Age и Fare по классам target — в один ряд
fig = make_subplots(rows=1, cols=2, subplot_titles=["Возраст (Age)", "Цена билета (Fare)"])
for status, color in colors.items():
    mask = survived_ru == status
    fig.add_trace(go.Box(x=survived_ru[mask], y=df.loc[mask, "Age"],
                         marker_color=color, showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(x=survived_ru[mask], y=df.loc[mask, "Fare"],
                         marker_color=color, showlegend=False), row=1, col=2)

fig.update_layout(
    template="plotly_white",
    height=400,
    title_text="Числовые признаки по классам target",
    title_x=0.5
)
fig.show()

Гипотеза: Fare и Pclass — почти один и тот же признак (класс выше т.к. билет дороже и наоборот). Проверка и вывод производного признака.

In [47]:
# Важная деталь: Fare в Titanic — цена за билет, который может быть куплен на группу.
# Поэтому напрашивается производный признак: цена на человека.
df["TicketGroupSize"] = df.groupby("Ticket")["Ticket"].transform("count")
df["FarePerPerson"] = df["Fare"] / df["TicketGroupSize"]

print("Медианы цен по классам:")
display(df.groupby("Pclass")[["Fare", "FarePerPerson"]].median().round(1))

print("Корреляция с Pclass (чем ближе к -1, тем сильнее связь «выше класс — дороже билет»):")
print("Fare:         ", df["Fare"].corr(df["Pclass"]).round(2))
print("FarePerPerson:", df["FarePerPerson"].corr(df["Pclass"]).round(2))

# Box plots в один ряд: сырой Fare и цена на человека, по классам
fig = make_subplots(rows=1, cols=2, subplot_titles=["Fare (цена за билет)", "FarePerPerson (цена на человека)"])
fig.add_trace(go.Box(x=df["Pclass"].astype(str), y=df["Fare"], marker_color="#4C78A8", showlegend=False), row=1, col=1)
fig.add_trace(go.Box(x=df["Pclass"].astype(str), y=df["FarePerPerson"], marker_color="#4C78A8", showlegend=False), row=1, col=2)

fig.update_layout(
    template="plotly_white",
    height=400,
    title_text="Цена билета относительно класса",
    title_x=0.5
)
fig.update_xaxes(title_text="Класс билета (Pclass)", categoryorder="category ascending")
fig.update_yaxes(title_text="Цена", row=1, col=1)
fig.show()

print("Можно далее работать с FarePerPerson вместо Fare")

Медианы цен по классам:


,Fare,FarePerPerson
Pclass,,
1,60.3,35.2
2,14.2,13.0
3,8.0,7.9


Корреляция с Pclass (чем ближе к -1, тем сильнее связь «выше класс — дороже билет»):
Fare:          -0.55
FarePerPerson: -0.66


Можно далее работать с FarePerPerson вместо Fare


In [42]:
import numpy as np

# FamilySize = SibSp + Parch + 1 (сам пассажир) — нужен для матрицы корреляции
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Корреляция выбранных числовых столбцов
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'FamilySize', 'Fare']
corr = df[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    template="plotly_white",
    width=750, height=650,
    title="Матрица корреляции признаков Titanic"
)
fig.update_traces(xgap=2, ygap=2)     # зазоры между ячейками
fig.update_xaxes(tickangle=45)        # поворот подписей по оси X
fig.update_yaxes(tickangle=-90)       # заголовки строк вертикально
fig.update_layout(
    title_x=0.5,
    margin=dict(l=10, r=10, t=50, b=10)  # меньше отступы до края
)
fig.show()


print('\n--- Оценка количества и размера семей ---')

# Только пассажиры, которые ехали не одни
df_families = df[df['FamilySize'] > 1]

# Метрики считаем по столбцу FamilySize отфильтрованных строк
min_family = df_families['FamilySize'].min()
max_family = df_families['FamilySize'].max()
mean_family = df_families['FamilySize'].mean().round(1)

family_counts = df_families['FamilySize'].value_counts().reset_index()
family_counts.columns = ['Размер семьи', 'Количество пассажиров']
family_counts = family_counts.sort_values(by='Размер семьи')


fig = px.bar(
    family_counts,
    x='Размер семьи',
    y='Количество пассажиров',
    title='Распределение размеров семей на Титанике (без одиноких пассажиров)',
    labels={'Размер семьи': 'Размер семьи (чел.)', 'Количество пассажиров': 'Число пассажиров'},
    text_auto=True,
    color='Количество пассажиров',
    color_continuous_scale='Cividis'
)

fig.update_layout(xaxis_dtick=1)
fig.show()

summary_table = pd.DataFrame({
    'Метрика': ['Минимальный размер семьи', 'Максимальный размер семьи', 'Средний размер семьи'],
    'Значение (чел.)': [min_family, max_family, mean_family],
    'Описание': ['Пара — двое едут вместе',
                 'Самая большая семья на борту (Sage)',
                 'Среднее по пассажирам с семьёй']
})
display(summary_table)


--- Оценка количества и размера семей ---


,Метрика,Значение (чел.),Описание
0,Минимальный размер семьи,2.0,Пара — двое едут вместе
1,Максимальный размер семьи,11.0,Самая большая семья на борту (Sage)
2,Средний размер семьи,3.3,Среднее по пассажирам с семьёй


Проверка двух гипотез:
1. Мальчики до 11 лет с родителем (Parch > 0) выживают на уровне женщин — против низкого показателя по всем мужчинам.
2. Большие семьи выживают хуже / лучше малых.

In [49]:
# --- Гипотеза 1: мальчики <= 11 лет с родителем выживают на уровне женщин ---
boys = df[(df["Sex"] == "male") & (df["Age"] <= 11) & (df["Parch"] > 0)]

groups = pd.DataFrame({
    "Группа": ["Мальчики ≤11 с родителем", "Все мужчины", "Все женщины"],
    "Человек": [len(boys),
                (df["Sex"] == "male").sum(),
                (df["Sex"] == "female").sum()],
    "Доля выживших": [boys["Survived"].mean().round(2),
                      df.loc[df["Sex"] == "male", "Survived"].mean().round(2),
                      df.loc[df["Sex"] == "female", "Survived"].mean().round(2)]
})
display(groups)

fig = px.bar(
    groups, x="Группа", y="Доля выживших",
    text_auto=".2f",
    color_discrete_sequence=["#4C78A8"],
    template="plotly_white", height=400,
    title="Доля выживших: мальчики с родителем против всех"
)
fig.update_layout(title_x=0.5, yaxis_range=[0, 1])
fig.show()

print(f"Вывод: мальчики ≤11 с родителем выживают в {groups['Доля выживших'][0] / groups['Доля выживших'][1]:.1f} раза чаще,")
print(f"чем мужчины в целом, и близко к уровню женщин. Но группа маленькая ({len(boys)} человек) —")
print("Age и Parch работают в связке с Sex, а не сами по себе.")

# --- Гипотеза 2: большие семьи выживают хуже / лучше малых ---
fam = df.groupby("FamilySize")["Survived"].agg(["count", "mean"]).reset_index() # странно звучит, что в семье из 11 человек только 7 человек - но это буквально 7 семей
fam.columns = ["Размер семьи", "Человек", "Доля выживших"]
fam["Доля выживших"] = fam["Доля выживших"].round(2)
display(fam)

fig = px.bar(
    fam, x="Размер семьи", y="Доля выживших",
    text_auto=".2f",
    color_discrete_sequence=["#4C78A8"],
    template="plotly_white", height=400,
    title="Доля выживших по размеру семьи"
)
fig.update_layout(title_x=0.5, xaxis_dtick=1, yaxis_range=[0, 1])
fig.show()

print("- малые семьи (2–4 человека) выживают лучше одиночек;")
print("- большие (5+) — резко хуже, хотя выборка небольшая")
print("Значит, FamilySize можно превратить в категорию: один / малая / большая")

,Группа,Человек,Доля выживших
0,Мальчики ≤11 с родителем,35,0.57
1,Все мужчины,577,0.19
2,Все женщины,314,0.74


Вывод: мальчики ≤11 с родителем выживают в 3.0 раза чаще,
чем мужчины в целом, и близко к уровню женщин. Но группа маленькая (35 человек) —
Age и Parch работают в связке с Sex, а не сами по себе.


,Размер семьи,Человек,Доля выживших
0,1,537,0.30
1,2,161,0.55
2,3,102,0.58
3,4,29,0.72
4,5,15,0.20
5,6,22,0.14
6,7,12,0.33
7,8,6,0.00
8,11,7,0.00


- малые семьи (2–4 человека) выживают лучше одиночек;
- большие (5+) — резко хуже, хотя выборка небольшая
Значит, FamilySize можно превратить в категорию: один / малая / большая


## Итого

- Данные: 891 строка, 12 столбцов; полных дубликатов нет, PassengerId уникален.
- Пропуски: Cabin ~77%, Age ~20%, Embarked — 2 строки.
- Пропуск в Cabin не случаен (MNAR): каюты записаны в основном у 1-го класса, поэтому сам факт «каюта известна» предсказывает выживание (~67% против ~30%) — это HasCabin.
- Target: умеренный дисбаланс классов (class imbalance) — 62% погибли / 38% выжили.
- Кардинальность: Name и PassengerId уникальны, Ticket почти уникален — как «сырые» категориальные признаки они бесполезны.
- Fare: есть перекос и много выбросов; явно связан с target (медиана у выживших ~26 против ~10).
- Fare и Pclass во многом описывают одно и то же («богатство» пассажира): медианы цен различаются по классам в разы. Fare — цена за билет на группу, поэтому выведен производный признак FarePerPerson = Fare / размер группы на билете: у него разброс внутри классов меньше и связь с Pclass сильнее.
- Age: сам по себе классы различает слабо — заметное различие только у детей.
- Sex: сильно связан с target — выжило ~74% женщин и ~19% мужчин.
- Мальчики ≤11 лет с родителем (Parch > 0) выживают близко к уровню женщин, а не мужчин — Age и Parch работают в связке с Sex, а не сами по себе. Группа маленькая, вывод статистически шаткий.
- Выживаемость по размеру семьи не линейная, а «горбом»: малые семьи (2–4) выживают лучше одиночек, большие (5+) — резко хуже.
- Корреляции: с target связаны Pclass (−0.34) и Fare (+0.26); SibSp и Parch коррелируют между собой (семьи ехали вместе).

Гипотезы о предсказательной силе (predictive power) признаков:

| Наблюдение | Гипотеза | Эксперимент |
|---|---|---|
| Age содержит ~20% пропусков | заполнение (imputation) улучшит модель | медиана vs другие способы |
| Fare перекошен | log1p поможет линейным моделям | с log1p vs без |
| Fare — цена за билет на группу | FarePerPerson чище, чем сырой Fare | Fare vs FarePerPerson |
| Sex сильно связан с target | обязательно использовать как признак | модель с ним vs без него |
| Мальчики с родителем выживают как женщины | признак «ребёнок» (Age ≤ 11) добавит сигнала | baseline vs с IsChild |
| Пропуск Cabin не случаен (MNAR) | факт «каюта известна» полезнее самого столбца | выбросить vs признак HasCabin |
| Pclass — всего 3 значения | и это категория, а не число | one-hot vs как число |
| Классы немного несбалансированы | веса классов (class weights) помогут | baseline vs weighted |
| Name содержит обращение (Mr/Mrs/Miss) | извлечь Title как новый признак (выживаемость одного супруга) | baseline vs с Title |
| SibSp/Parch описывают семью | собрать FamilySize | baseline vs с FamilySize |
| Выживаемость по FamilySize — «горбом» | категории один/малая/большая лучше числа | FamilySize число vs категории |